In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np

FILE_PATH = "/content/drive/MyDrive/KB_FinalProject_Team6/인허가_분석결과/인허가_기본16컬럼_20261028_100154.csv"

# ID와 행정동코드 보존
df_base = pd.read_csv(
    FILE_PATH,
    encoding="utf-8-sig",
    dtype="string"
)

df = df_base.copy()

# CSV에는 자료형이 보존되지 않으므로 복원
df["인허가일자_dt"] = pd.to_datetime(
    df_base["인허가일자_dt"],
    format="ISO8601",
    errors="coerce"
)

for col in ["lon", "lat", "프랜차이즈유무"]:
    df[col] = pd.to_numeric(df_base[col], errors="coerce")

# 변환 중 새로 생긴 결측 확인
converted_cols = ["인허가일자_dt", "lon", "lat", "프랜차이즈유무"]

for col in converted_cols:
    failed = df_base[col].notna() & df[col].isna()
    print(f"{col} 변환 실패: {failed.sum():,}건")

print(f"\n데이터: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")
display(df.head())

In [ ]:
# 점포이력ID별 서로 다른 업종 수
biz_counts = df.groupby("점포이력ID")["업종명"].nunique()

multi_biz_ids = biz_counts[biz_counts >= 2].index

# 해당 점포의 모든 행·컬럼 조회
multi_biz_all = (
    df.loc[df["점포이력ID"].isin(multi_biz_ids)]
    .sort_values(["점포이력ID", "인허가일자_dt", "업종명"])
    .copy()
)

print(f"해당 점포: {len(multi_biz_ids):,}개")
print(f"해당 인허가 행: {len(multi_biz_all):,}건")

# 모든 컬럼 표시, 행은 우선 50개 확인
with pd.option_context("display.max_columns", None):
    display(multi_biz_all.head(50))

In [ ]:
multi_biz_rows = (
    df.loc[
        df["점포이력ID"].isin(multi_biz_stores["점포이력ID"]),
        [
            "점포이력ID", "점포ID", "사업장명",
            "업종명", "카드업종", "업태구분명",
            "구군", "법정동", "lon", "lat",
            "인허가일자_dt"
        ]
    ]
    .sort_values(["점포이력ID", "인허가일자_dt", "업종명"])
)

display(multi_biz_rows.head(50))

In [ ]:
from pathlib import Path

matches = list(
    Path("/content/drive/MyDrive").rglob("인허가폐업일자빼오기용.csv")
)

for i, path in enumerate(matches):
    print(i, path)

print(f"\n발견된 파일: {len(matches)}개")

In [ ]:
close_info = pd.read_csv(
    matches[0],
    encoding="cp949",
    dtype="string",
    usecols=["점포ID", "폐업일자", "영업여부"]
)

close_info["점포ID"] = close_info["점포ID"].str.strip()

print(f"불러오기 완료: {len(close_info):,}행")
display(close_info.head())

In [ ]:
print(f"행 수: {close_info.shape[0]:,}")
print(f"컬럼 수: {close_info.shape[1]}")
print("컬럼 목록:", close_info.columns.tolist())

display(close_info.head(10))
close_info.info()

In [ ]:
report = pd.DataFrame({
    "자료형": close_info.dtypes.astype(str),
    "결측수": close_info.isna().sum(),
    "결측률(%)": close_info.isna().mean().mul(100).round(2),
    "고유값수": close_info.nunique()
})

display(report)

In [ ]:
ids = close_info["점포ID"].astype("string").str.strip()

print("점포ID 결측:", ids.isna().sum())
print("점포ID 빈 문자열:", ids.eq("").sum())
print("점포ID 중복 초과 행:", ids.duplicated().sum())

# 중복 ID가 있으면 해당 행 전체 확인
duplicate_mask = ids.notna() & ids.duplicated(keep=False)

display(
    close_info.loc[duplicate_mask]
    .sort_values("점포ID")
)

In [ ]:
date_text = close_info["폐업일자"].astype("string").str.strip()
has_close_date = date_text.notna() & date_text.ne("").fillna(False)

print("영업여부별 건수")
display(close_info["영업여부"].value_counts(dropna=False))

print("영업여부 × 폐업일자 유무")
display(
    pd.crosstab(
        close_info["영업여부"].fillna("상태 결측"),
        has_close_date.map({True: "폐업일자 있음", False: "폐업일자 없음"})
    )
)

In [ ]:
parsed_dates = pd.to_datetime(
    date_text,
    format="ISO8601",
    errors="coerce"
)

failed = has_close_date & parsed_dates.isna()

print("날짜 변환 실패:", failed.sum())
print("최초 폐업일:", parsed_dates.min())
print("최근 폐업일:", parsed_dates.max())

# 시간은 제외하고 날짜별 집계
display(
    parsed_dates.dt.normalize()
    .value_counts()
    .head(20)
    .rename_axis("폐업일자")
    .reset_index(name="건수")
)

if failed.any():
    display(close_info.loc[failed])

In [ ]:
our_ids = df["점포ID"].astype("string").str.strip()
close_ids = close_info["점포ID"].astype("string").str.strip()

matched = our_ids.notna() & our_ids.isin(close_ids.dropna())

print(f"우리 df: {len(df):,}행")
print(f"폐업 정보: {len(close_info):,}행")
print(f"매칭 성공: {matched.sum():,}행")
print(f"매칭 실패: {(~matched).sum():,}행")
print(f"폐업 정보 점포ID 중복: {close_ids.duplicated().sum():,}행")

display(
    df.loc[
        ~matched,
        ["점포ID", "점포이력ID", "사업장명"]
    ].head(30)
)

In [ ]:
# 재실행해도 동일 컬럼이 중복 생성되지 않도록 처리
left = df.drop(
    columns=["폐업일자", "영업여부"],
    errors="ignore"
).copy()

right = close_info[
    ["점포ID", "폐업일자", "영업여부"]
].copy()

# 연결 키 형식 통일
left["점포ID"] = left["점포ID"].astype("string").str.strip()
right["점포ID"] = right["점포ID"].astype("string").str.strip()

# 점포ID가 같은 행끼리 연결
merged = left.merge(
    right,
    on="점포ID",
    how="left",
    validate="many_to_one",
    indicator="_매칭결과",
    sort=False
)

# 연결 결과 검증
assert len(merged) == len(df), "연결 전후 행 수가 다릅니다."
assert merged["_매칭결과"].eq("both").all(), "매칭 실패 행이 있습니다."

# 검증 통과 후 df에 반영
df = merged.drop(columns="_매칭결과").copy()

print(f"연결 완료: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")

display(
    df[["점포ID", "사업장명", "폐업일자", "영업여부"]].head(20)
)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
# 빈 문자열도 결측으로 인식하는 변환용 데이터
date_text = (
    df["폐업일자"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df["폐업일자_dt"] = pd.to_datetime(
    date_text,
    format="ISO8601",
    errors="coerce"
)

# 원래 날짜 값이 있었는데 변환에 실패한 경우
failed = date_text.notna() & df["폐업일자_dt"].isna()

print("날짜 변환 실패:", failed.sum())
print("최초 폐업일:", df["폐업일자_dt"].min())
print("최근 폐업일:", df["폐업일자_dt"].max())

display(
    df.loc[failed, ["점포ID", "사업장명", "폐업일자"]]
)

In [ ]:
status = df["영업여부"].astype("string").str.strip()

date_state = pd.Series("날짜 없음", index=df.index)
date_state.loc[df["폐업일자_dt"].notna()] = "날짜 있음"
date_state.loc[failed] = "변환 실패"

display(
    pd.crosstab(
        status.fillna("상태 결측"),
        date_state,
        rownames=["영업여부"],
        colnames=["폐업일자 상태"]
    )
)

In [ ]:
issues = pd.DataFrame({
    "날짜변환실패": failed,
    "영업인데_폐업일자있음": status.eq("T") & date_text.notna(),
    "폐업인데_폐업일자없음": status.eq("F") & date_text.isna(),
    "영업상태_결측또는기타": ~status.isin(["T", "F"]),
    "인허가일보다_폐업일이빠름": (
        df["폐업일자_dt"] < df["인허가일자_dt"]
    )
}).fillna(False)

display(issues.sum().rename("건수").to_frame())

review_rows = df.loc[
    issues.any(axis=1),
    [
        "점포ID", "점포이력ID", "사업장명",
        "인허가일자_dt", "폐업일자",
        "폐업일자_dt", "영업여부"
    ]
]

display(review_rows)

In [ ]:
if "폐업일자_dt" in df.columns:
    print("폐업일자_dt 자료형:", df["폐업일자_dt"].dtype)
    display(df[["폐업일자", "폐업일자_dt"]].head(10))
else:
    print("현재 df에 폐업일자_dt 컬럼이 없습니다.")

In [ ]:
df

In [ ]:
from pathlib import Path

save_dir = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/인허가_분석결과"
)
save_dir.mkdir(parents=True, exist_ok=True)

stamp = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y%m%d_%H%M%S")
save_path = save_dir / f"인허가_폐업정보결합_EDA준비_{stamp}.csv"

df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"저장 완료: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")
print("파일 경로:", save_path)